# Bronze proofs
1. Schema (explicit StructType) 2. Rerun = no duplicates 3. Schema drift → quarantine 4. Failure gets logged

In [0]:
%run ./bronze_functions

In [0]:
%sql
DESCRIBE TABLE workspace.bronze.mkt_summary

In [0]:
def snapshot():
    return spark.sql("""
        SELECT file_trade_date, COUNT(*) AS n, MAX(batch_id) AS batch_id
        FROM workspace.bronze.mkt_summary
        WHERE file_trade_date BETWEEN '2015-01-01' AND '2015-01-09'
        GROUP BY file_trade_date ORDER BY file_trade_date
    """).toPandas()

before = snapshot()
display(before)

In [0]:
rerun_batch = load_range("2015-01-01", "2015-01-09", "backfill")

In [0]:
after = snapshot()
display(after)

cmp = before.merge(after, on="file_trade_date", suffixes=("_before", "_after"))
assert (cmp["n_before"] == cmp["n_after"]).all(), "Row counts changed!"
assert (after["batch_id"] == rerun_batch).all(), "Old rows were not replaced"

dups = spark.sql("""
    SELECT file_trade_date, symbol, COUNT(*) c FROM workspace.bronze.mkt_summary
    GROUP BY file_trade_date, symbol HAVING COUNT(*) > 1
""").count()
assert dups == 0
print(f"PASS: {after['n'].sum()} rows before and after, every row now from {rerun_batch}, 0 duplicates")

In [0]:
%sql
DESCRIBE HISTORY workspace.bronze.mkt_summary LIMIT 10

In [0]:
real = "/Volumes/workspace/bronze/raw_landing/mkt_summary/2015/2015-01-02.Z"
fake = "/Volumes/workspace/bronze/raw_landing/mkt_summary/2015/2015-01-03.Z"   # Saturday: no real file

raw = open(real, "rb").read()
if raw[:2] == b"PK":
    with zipfile.ZipFile(io.BytesIO(raw)) as z:
        text = z.read(z.namelist()[0]).decode("latin-1")
else:
    text = gzip.decompress(raw).decode("latin-1")

lines = text.splitlines()
for i in range(5):
    lines[i] += "|NEW_COLUMN"          # 13 pipes instead of 12 = drift

with open(fake, "wb") as f:
    f.write(gzip.compress("\n".join(lines).encode("latin-1")))
print("Fake file written with", len(lines), "lines, 5 drifted")

In [0]:
test_batch = f"bronze_test_{dt.datetime.now().strftime('%Y-%m-%d_%H-%M-%S')}"
start = dt.datetime.now()
rows, quarantined = load_to_bronze("2015-01-03", test_batch, "backfill")
write_log(test_batch, "raw_to_bronze", "backfill", fake, start, dt.datetime.now(),
          "SUCCESS", rows_inserted=rows, rows_quarantined=quarantined)
print(f"Loaded {rows}, quarantined {quarantined}")

In [0]:
%sql
SELECT reason, raw_record, source_file, batch_id FROM workspace.ops.quarantine_mkt_summary
WHERE file_trade_date = '2015-01-03'

In [0]:
bad = "/Volumes/workspace/bronze/raw_landing/mkt_summary/2015/2015-01-04.Z"
with open(bad, "wb") as f:
    f.write(b"this is not a real archive")

start = dt.datetime.now()
try:
    load_to_bronze("2015-01-04", test_batch, "backfill")
except Exception as e:
    write_log(test_batch, "raw_to_bronze", "backfill", bad, start, dt.datetime.now(),
              "FAILURE", error_message=str(e)[:1000])
    print("Caught and logged:", str(e)[:200])

In [0]:
%sql
SELECT parameter, status, rows_inserted, rows_quarantined, error_message
FROM workspace.ops.pipeline_execution_logs
WHERE batch_id LIKE 'bronze_test_%' ORDER BY start_time

In [0]:
spark.sql("DELETE FROM workspace.bronze.mkt_summary WHERE file_trade_date IN ('2015-01-03','2015-01-04')")
spark.sql("DELETE FROM workspace.ops.quarantine_mkt_summary WHERE file_trade_date = '2015-01-03'")
dbutils.fs.rm(fake); dbutils.fs.rm(bad)
print("Fake files and rows removed")

In [0]:
%sql
   SELECT COUNT(*) FROM workspace.bronze.mkt_summary
   WHERE file_trade_date IN ('2015-01-03','2015-01-04')